In [ ]:
import scipy
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

from sklearn import linear_model    # Herramientas de modelos lineales
from sklearn.metrics import mean_squared_error, r2_score    # Medidas de desempeño
from sklearn.preprocessing import PolynomialFeatures    # Herramientas de polinomios

from formulaic import Formula

In [ ]:
# Si necesitan instalar algún paquete
#pip install gapminder
#pip install formulaic

## Introducción: sistemas de ecuaciones lineales

Sabemos que un fondo de inversión invirtió en acciones de YPF, Santander y Nvidia (y solo en estas acciones) pero no sabemos cuántas acciones compró de cada una. ¿Cómo podemos averiguarlo? 

Suponemos que tenemos disponible:
1. La valorización del fondo al final de cada día.
2. El valor la acción de cada empresa al cierre de cada día.

In [ ]:
# Cargamos los datos
dataDict = {'total': [170262,169929.5,171064,169637.35,164625.45], 
        'YPF': [20935, 21030, 20770, 20950, 20750], 
        'Santander': [20100, 20500, 21700, 21000, 20316], 
        'Nvidia': [37100, 36255, 36000, 35645.5, 33878.5]}
data = pd.DataFrame.from_dict(dataDict)
data

Nos quedamos con las primeras tres filas y resolvemos el sistema lineal:
$$total = c_1 \cdot YPF + c_2 \cdot Santander + c_3 \cdot Nvidia$$


In [ ]:
data_3rows = data[[True, True, True, False, False]]
data_3rows

In [ ]:
X_3rows = data_3rows[["YPF", "Santander", "Nvidia"]]
y_3rows = data_3rows["total"]
display(X_3rows)
display(y_3rows)

Para obtener los valores de c_1, c_2 y c_3 resolvemos el sistema lineal utilizando `np.linalg.solve`

In [ ]:
c = np.linalg.solve(X_3rows, y_3rows)
print(c)

In [ ]:
# Verificamos
X_3rows @ c

In [ ]:
# Verificamos que se satisface también las otras ecuaciones
X = data[["YPF", "Santander", "Nvidia"]]
X @ c

In [ ]:
# Como verificamos si coincide con los totales que teníamos?

# Caso de estudio: calorías de alimentos

In [ ]:
df_nutricion = pd.read_csv('nutrition.csv')
df_nutricion

Vemos que el DataFrame contiene muchos datos "NaN" (not a number). ¿En qué columnas hay NaN? Qué estrategia adoptamos?

In [ ]:
df_nutricion.head()

Proponemos un modelo lineal, donde la variable respuesta es combinación lineal de todas las otras variables.

Construimos las matrices X e y utilizando Formulaic

In [ ]:
# Utilizamos - 1 para no incluir el intercept
y, X = (
    Formula('Calorias_kcal ~ Proteinas_g + Carbohidratos_g + GrasaTotal_g + Colesterol_mg + Fibra_g + Agua_g + Alcohol_g + VitaminaC_mg  - 1')
    .get_model_matrix(df_nutricion)
)

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Formulaic nos devuelve un DataFrame para y. Para muchas cosas es mejor tener a y como una serie
y = y.squeeze() # Squeeze reduce las dimensiones que puede, sin asumir nada
y

**Observación**

En este caso sencillo, podemos obtener lo mismo (excepto por la columna Intercept) utilizando una lista de columnas. O mejor: usar `.drop()`


In [ ]:
X_columns = ["Proteinas_g", "Carbohidratos_g", "GrasaTotal_g", "Colesterol_mg","Fibra_g", "Agua_g","Alcohol_g","VitaminaC_mg"]
X = ???
y = ???

Ajustamos el modelo lineal.

**Pregunta:** Es razonable en este caso usar Intercept en el modelo?

In [ ]:
# fit_intecept True o False?
modelo = linear_model.LinearRegression(fit_intercept = ???)    # Inicializamos un modelo de Regresion Lineal.   
modelo.fit(X, y)   # Realizamos el ajuste

Analizamos la "bondad" del ajuste.

In [ ]:
y_pred = modelo.predict(X)
# Calculando el R^2
r2 = r2_score(y, y_pred)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(y, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

A priori es un buen modelo, tenemos 7792 observaciones y obtenemos R^2 casi igual a 1 con solo 9 variables.

**Preguntas:** 
1. ¿Qué hace `fit` y qué hace `predict`? 
2. Hay un método `fit_predict()` qué hace las dos cosas de una, ¿cuándo puede ser útil hacer las cosas por separado?

Analizamos el modelo y veamos si podemos obtener una fórmula práctica.

In [ ]:
modelo.coef_

**Ejercicio:** Cómo podemos obtener estos coeficientes usando la función `np.solve`?

In [ ]:
# Recordemos que para minimizar el error de Xc = y, tenemos que resolver X^T X c = X^T y


Analizando los coeficientes, ¿qué variables tienen mayor peso en el modelo?
Veamos si podemos eliminar las demás variables.

**Spoiler 1:** No podemos determinar la importancia de las variables en el modelo solo mirando los coeficientes, porque no sabemos si las variables están en escalas similares (o si hay dependencia entre las varaibles). Necesitamos primero normalizar las variables para hacer este análisis! 

In [ ]:
# Igualmente, probemos y veamos qué pasa...
y2, X2 = (
    Formula('Calorias_kcal ~ Proteinas_g + Carbohidratos_g + GrasaTotal_g + ??? - 1')
    .get_model_matrix(df_nutricion)
)

In [ ]:
modelo = linear_model.LinearRegression()    # Inicializamos un modelo de Regresion Lineal. 
   
modelo.fit(X2, y2)   # Realizamos el ajuste

In [ ]:
y_pred = modelo.predict(X2)
# Calculando el R^2
r2 = r2_score(y, y_pred)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(y, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

In [ ]:
modelo.coef_

In [ ]:
# Y si eliminamos el intercept?
modelo = linear_model.LinearRegression(???)    # Inicializamos un modelo de Regresion Lineal. 
modelo.fit(X2, y2)   # Realizamos el ajuste

In [ ]:
y_pred = modelo.predict(X2)
# Calculando el R^2
r2 = r2_score(y, y_pred)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(y, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

**Spoiler 2**: ¿cómo elegimos con cuál modelo quedarnos? Un modelo con más variables va a dar siempre un error menor en los datos, pero eso ¿significa que el modelo es mejor? ¿Podemos asegurar que en otros datos nuevos también va a dar mejor?

Para ver cuál predice mejor tenemos que diseñar un esquema de validación de modelos, separar los datos en entrenamiento y testeo.

### Ejercicio
1. Construir un modelo lineal que explique lo mejor posible la variable `mpg` utilizando las variables numéricas disponibles.
2. El comando `sns.pairplot()` realiza automáticamente una grilla de gráficos entre pares de variables en un DataFrame. A partir de esos gráficos, si tuvieran que elegir solo 3 variables explicativas para modelar `mpg`, cuáles usarían?

In [ ]:
mpg = sns.load_dataset('mpg')
mpg

In [ ]:
sns.pairplot(mpg)

### Ejercicio

Trabajamos con el DataSet `diamonds` incluido en SNS.

1. Construir un modelo lineal que explique lo mejor posible la variable `price` utilizando las demás variables numéricas disponibles.
2. Observar los coeficientes obtenidos. Notan algo raro?
3. Utilizar `sns.pairplot()` para entender qué está pasando. A partir de esos gráficos, si quisieran utilizar pocas variables explicativas para modelar `price`, cuáles usarían?

In [ ]:
diamonds = sns.load_dataset('diamonds')
diamonds

**Ejercicio**
Para los datos del fondo de inversión, encontrar los coeficientes del modelo usando todas las filas, no solo las tres primeras.

In [ ]:
# Cargamos los datos
dataDict = {'total': [170262,169929.5,171064,169637.35,164625.45], 
        'YPF': [20935, 21030, 20770, 20950, 20750], 
        'Santander': [20100, 20500, 21700, 21000, 20316], 
        'Nvidia': [37100, 36255, 36000, 35645.5, 33878.5]}
data = pd.DataFrame.from_dict(dataDict)
data